# Diagnostic : d'où viennent les erreurs ? (sans GPU)

Avant de mobiliser des lecteurs ou des locuteurs, savoir ce qui limite
l'assistant : **la reconnaissance** (Soloni entend mal) ou **la traduction**
(il entend bien, mais traduit mal). Rien n'est entraîné ici.

1. **Les erreurs de Soloni sur ta voix** : tes lectures, dont le texte exact
   est connu (taux d'erreur par mot, et dix exemples).
2. **Ton jugement de 100 phrases**, deux questions chacune : le sens est-il
   compris ? Ce que Soloni a écrit en bambara est-il bien ce que tu as dit ?

**Sans GPU** : Exécution → Modifier le type d'exécution → **CPU**. Le quota
de GPU n'est pas touché. Exécution → **Tout exécuter**, autoriser Drive ;
compter 20 à 30 minutes d'installation et de calcul, puis le jugement
(30 à 40 minutes, les boutons répondent quand tout le reste est fini).

À la fin, m'envoyer les trois chiffres : le taux d'erreur de Soloni, le
partage des erreurs (reconnaissance / traduction), le taux de compréhension.

## Installation et Drive

In [ ]:
# Branche de travail : contient les dernières corrections du notebook et des
# scripts. Une fois fusionnée, la branche par défaut suffit (sans -b).
BRANCH = "claude/chat-ia-bambara-dho8oa"

import os
os.chdir("/content")
if os.path.isdir("projet/.git"):
    !git -C projet pull -q   # relance : code mis à jour, rien à recloner
else:
    !git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd /content/projet
!pip install -q transformers datasets accelerate sentencepiece jiwer sacrebleu soundfile librosa

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT = "/content/drive/MyDrive/bambara/modeles"
OUT = "/content/drive/MyDrive/bambara/resultats-phase3"
ATELIER = "/content/drive/MyDrive/bambara/atelier"

In [ ]:
# Modèle final s'il existe, sinon le dernier checkpoint (entraînement coupé
# ou inachevé), sinon le modèle d'origine quand il y en a un. Défini ici pour
# rester disponible après une reprise (sections 1 à 3) ; les sections 6 à 10
# appellent chemins().
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

def modele(dossier, repli=None):
    if (Path(dossier) / "model.safetensors").exists():
        return dossier
    dernier = get_last_checkpoint(dossier) if Path(dossier).is_dir() else None
    if dernier:
        print(f"Entraînement inachevé : {dernier}")
        return dernier
    if repli:
        print(f"Pas encore entraîné ({dossier}) : modèle d'origine {repli}")
        return repli
    raise FileNotFoundError(f"ni modèle final ni checkpoint dans {dossier}")

def deja(label):
    """Vrai si un rapport porte déjà ce nom : relancer une cellule
    d'évaluation, d'une session à l'autre, ne la refait pas. Sauf si l'un
    de ses modèles a été réentraîné depuis : l'ancien rapport est alors mis
    de côté (.ancien) et l'évaluation refaite."""
    import glob, json, os
    for p in glob.glob(f"{OUT}/*.json"):
        try:
            rapport = json.load(open(p))
        except ValueError:  # copie vers Drive interrompue : rapport tronqué
            os.rename(p, p + ".tronque")
            continue
        if rapport.get("nom") != label:
            continue
        poids = [f for m in rapport.get("modeles", {}).values() if isinstance(m, str)
                 for f in glob.glob(f"{m}/*.safetensors")]
        if any(os.path.getmtime(f) > os.path.getmtime(p) for f in poids):
            for ancien in (p, p[:-len(".json")] + ".details.jsonl"):
                if os.path.exists(ancien):
                    os.rename(ancien, ancien + ".ancien")
            print(f"« {label} » évalué avec un modèle réentraîné depuis : refait.")
            return False
        print(f"« {label} » déjà évalué ({p}) : cellule sautée.")
        return True
    return False

def a_jour(motif, code=False):
    """Vrai si la sortie la plus récente qui correspond au motif est
    postérieure aux poids de tous les modèles en service (chemins()) et,
    avec code=True, au dernier commit qui modifie la chaîne (bambara_voice) :
    sinon, l'étape est à refaire."""
    import glob, os, subprocess
    sorties = glob.glob(motif)
    if not sorties:
        return False
    modeles = {*chemins(), *meilleurs()}
    reperes = [os.path.getmtime(f) for m in modeles for f in glob.glob(f"{m}/*.safetensors")]
    if code:
        commit = subprocess.run(["git", "log", "-1", "--format=%ct", "--", "bambara_voice"],
                                capture_output=True, text=True).stdout.strip()
        reperes += [float(commit)] if commit else []
    derniere = max(os.path.getmtime(f) for f in sorties)
    return all(r < derniere for r in reperes)

def place_drive(besoin_go):
    """Arrête la cellule si Drive n'a pas la place : un Drive plein ne
    reçoit plus rien, sans aucune erreur dans Colab."""
    import shutil
    libre = shutil.disk_usage("/content/drive").free / 1e9
    print(f"Place libre sur Drive : {libre:.1f} Go (il en faut {besoin_go})")
    if libre < besoin_go:
        raise RuntimeError("Drive presque plein. Les fichiers supprimés restent dans la "
                           "corbeille et comptent encore : la vider sur drive.google.com.")

def synchroniser_drive(minutes=30):
    """Attend que Drive ait reçu tout ce qui a été écrit, puis le remonte
    (Colab peut redemander l'accès : accepter la fenêtre qui s'ouvre).
    Ne lève jamais d'erreur : « Tout exécuter » continue."""
    try:
        drive.flush_and_unmount(timeout_ms=minutes * 60 * 1000)
        print("Drive a tout reçu.")
    except Exception as e:
        print(f"Drive n'a pas tout reçu en {minutes} min ({e}). Drive plein ? Vider la "
              "corbeille sur drive.google.com, puis relancer cette cellule.")
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive non remonté ({e}) : relancer la section 2 avant la démo.")

def chemins():
    """Whisper, NLLB bambara → français et NLLB français → bambara à
    utiliser, d'après ce qui est sur Drive."""
    nllb = "facebook/nllb-200-distilled-600M"
    return (modele(f"{CKPT}/whisper-small-bm"),
            modele(f"{CKPT}/nllb-bm2fr", repli=nllb),
            modele(f"{CKPT}/nllb-fr2bm", repli=nllb))

RETENU = f"{CKPT}/retenu.json"           # l'assemblage en service et ses preuves
MOYENNE = "/content/nllb-bm2fr-moyenne"  # moyenne des poids v1/v2 (§9a), refaite au besoin

def en_service():
    """L'assemblage en service (eval.promotion) : Whisper + NLLB v1, celui de
    la démo du 15 octobre, tant qu'aucun candidat n'a fait mieux, mesure à
    l'appui, sur le test maison. Un nouveau modèle ne le remplace que là."""
    from eval.promotion import lire_retenu
    whisper, bm2fr, fr2bm = chemins()
    r = lire_retenu(RETENU, {"asr": {"model_id": whisper}, "nom_asr": "Whisper",
                             "bm2fr": bm2fr, "nom_bm2fr": "NLLB v1",
                             "fr2bm": fr2bm, "nom_fr2bm": "NLLB v1"})
    if r["bm2fr"] == MOYENNE and not Path(f"{MOYENNE}/model.safetensors").exists():
        import subprocess  # la moyenne vit sur le disque de Colab : une minute
        subprocess.run(["python", "scripts/moyenne_poids.py", "--a", bm2fr,
                        "--b", f"{CKPT}/nllb-bm2fr-v2", "--out", MOYENNE], check=True)
    return r

def meilleurs():
    """Les modèles de l'assistant (§9 à 11) : Whisper (v2 si entraîné) et les
    deux NLLB en service."""
    r = en_service()
    whisper = chemins()[0]
    if Path(f"{CKPT}/whisper-small-bm-v2/model.safetensors").exists():
        whisper = f"{CKPT}/whisper-small-bm-v2"
    return whisper, r["bm2fr"], r["fr2bm"]

## Ce qui est en service

In [ ]:
import glob, json, os
from eval.promotion import score
WHISPER, NLLB_BM2FR, NLLB_FR2BM = meilleurs()
EN_SERVICE = en_service()
CFG = "/content/assistant-affine.json"
import torch  # sans GPU, sur processeur : plus lent, mais sans quota
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
json.dump({"device": DEVICE, "asr": EN_SERVICE["asr"], "mt_in": {"model_id": NLLB_BM2FR},
           "mt_out": {"model_id": NLLB_FR2BM}}, open(CFG, "w"), indent=1)
MAISON, PRISES = f"{ATELIER}/testset-dire.jsonl", f"{ATELIER}/dire.jsonl"
TITULAIRE = f"{EN_SERVICE['nom_asr']} + {EN_SERVICE['nom_bm2fr']}"
RAPPORT_EN_SERVICE = None
if not os.path.exists(PRISES):
    print("Aucune phrase enregistrée avec l'atelier : la v4 ne peut pas être jugée.")
else:
    if EN_SERVICE["asr"].get("kind") == "nemo":   # Soloni : jugement de la v4 au §5
        !pip install -q "nemo_toolkit[asr]"
    !python -m eval.atelier --racine "$ATELIER" exporter > /dev/null
    dossier = f"{OUT}/maison-assemblages/{EN_SERVICE['nom_asr']}-{EN_SERVICE['nom_bm2fr']}".replace(" ", "-")
    motif = f"{dossier}/cascade-*[0-9].json"
    if not (glob.glob(motif) and max(map(os.path.getmtime, glob.glob(motif))) > os.path.getmtime(PRISES)):
        print(f"{TITULAIRE} : mesure sur tes phrases…")
        !python -m eval.run_eval --testset "$MAISON" --arch cascade --config $CFG \
            --comprendre-seulement --nom "$TITULAIRE" --out "$dossier" > /dev/null 2>&1
    if glob.glob(motif):
        RAPPORT_EN_SERVICE = sorted(glob.glob(motif))[-1]
        print(f"En service : {TITULAIRE} ; chrF++ sur ta voix : dev "
              f"{score(RAPPORT_EN_SERVICE, 'dev'):.1f}, test {score(RAPPORT_EN_SERVICE, 'test'):.1f}")
    else:
        print(f"⚠️ {TITULAIRE} n'a pas pu être mesuré : relancer la cellule.")

## 1. Combien Soloni se trompe sur ta voix (tes lectures)

Le WER compte les mots mal reconnus sur 100. « Relâché » ne compte pas les
variantes d'orthographe (e/ɛ, o/ɔ). Repère : sur les corpus de RobotsMali,
Soloni est autour de 30 à 40 % ; nettement au-dessus sur ta voix, c'est la
reconnaissance qu'il faudra adapter à ta voix ensuite (avec ces mêmes
lectures).

In [ ]:
import glob, json, os
LECTURES, PRISES_LIRE = f"{ATELIER}/testset-lire.jsonl", f"{ATELIER}/lire.jsonl"
if not (os.path.exists(PRISES_LIRE) and os.path.exists(LECTURES) and os.path.getsize(LECTURES)):
    print("Aucune lecture enregistrée : mesure sautée.")
else:
    dossier_lu = f"{OUT}/lectures/{EN_SERVICE['nom_asr']}".replace(" ", "-")
    motif_lu = f"{dossier_lu}/cascade-*[0-9].json"
    if not (glob.glob(motif_lu) and max(map(os.path.getmtime, glob.glob(motif_lu)))
            > os.path.getmtime(PRISES_LIRE)):
        print(f"{EN_SERVICE['nom_asr']} : reconnaissance de tes lectures…")
        !python -m eval.run_eval --testset "$LECTURES" --arch cascade --config $CFG \
            --comprendre-seulement --nom "$TITULAIRE" --out "$dossier_lu" > /dev/null 2>&1
    if glob.glob(motif_lu):
        lu = sorted(glob.glob(motif_lu))[-1]
        a = json.load(open(lu))["asr"]
        print(f"{EN_SERVICE['nom_asr']} sur tes {a['n']} lectures : WER {100 * a['wer']:.1f} %, "
              f"relâché {100 * a['wer_folded']:.1f} % (lettres : {100 * a['cer_folded']:.1f} %)\n")
        exemples = [json.loads(l) for l in open(lu.replace(".json", ".details.jsonl"))][:10]
        for r in exemples:
            print(f"lu      : {r['ref_transcript_bm']}\nentendu : {r.get('asr_text') or '(rien)'}\n")
    else:
        print("⚠️ Mesure des lectures échouée : relancer la cellule.")

## 2. Ton jugement : l'assistant a-t-il compris ?

100 phrases que tu as dites, toujours les mêmes, et pour chacune deux
questions :

1. **Le sens** : la phrase affichée, et ce que l'assistant a compris. Note le
   sens, pas les mots.
2. **Ce que Soloni a entendu**, écrit en bambara : est-ce bien ce que tu as
   dit ? Tu sais lire le bambara : c'est ce qui dira si les erreurs viennent
   de la reconnaissance (il a mal entendu) ou de la traduction (il a bien
   entendu, mais mal traduit), et donc s'il faut des lectures ou des phrases
   dites.

Sans GPU : cette partie marche aussi sur CPU, un autre jour. Tes notes sont
gardées au fil de l'eau ; relancer la cellule reprend où tu t'étais arrêté.

Les boutons ne répondent qu'une fois toutes les cellules précédentes
terminées.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display
from eval.comprehension import a_juger, bilan, noter, to_markdown
NOTES_COMPREHENSION = f"{ATELIER}/comprehension.jsonl"
restants = a_juger(RAPPORT_EN_SERVICE, NOTES_COMPREHENSION, TITULAIRE) if RAPPORT_EN_SERVICE else []
ecran = widgets.HTML()
boutons = [widgets.Button(layout=widgets.Layout(width="150px")) for _ in range(3)]
LIBELLES = {"sens": ("✅ Même sens", "🟡 À moitié", "❌ Faux"),
            "entendu": ("✅ Bien entendu", "🟡 À peu près", "❌ Mal entendu")}
etat = {"question": "sens", "sens": None}

def montrer():
    if not restants:
        ecran.value = "<pre>" + to_markdown(bilan(NOTES_COMPREHENSION, TITULAIRE)) + "</pre>"
        for b in boutons:
            b.disabled = True
        return
    r = restants[0]
    if etat["question"] == "sens":
        ecran.value = (f"<p>Encore {len(restants)} phrases — 1/2 : le sens</p>"
                       f"<p><b>Tu as dit :</b> {html.escape(r['dit'])}</p>"
                       f"<p><b>L'assistant a compris :</b> {html.escape(r['compris']) or '(rien)'}</p>")
    else:
        ecran.value = (f"<p>Encore {len(restants)} phrases — 2/2 : ce que Soloni a entendu</p>"
                       f"<p><b>Tu voulais dire :</b> {html.escape(r['dit'])}</p>"
                       f"<p><b>Soloni a écrit :</b> {html.escape(r['entendu']) or '(rien)'}</p>"
                       "<p>Est-ce bien ce que tu as dit en bambara ?</p>")
    for b, libelle in zip(boutons, LIBELLES[etat["question"]]):
        b.description = libelle

def repondre(valeur):
    def clic(_):
        if etat["question"] == "sens":
            etat.update(question="entendu", sens=valeur)
        else:
            noter(NOTES_COMPREHENSION, restants.pop(0)["item"], etat["sens"], TITULAIRE, valeur)
            etat.update(question="sens", sens=None)
        montrer()
    return clic

for bouton, valeur in zip(boutons, (2, 1, 0)):
    bouton.on_click(repondre(valeur))
montrer()
display(widgets.VBox([ecran, widgets.HBox(boutons)]))